# 03 - PINN 反演热方程中的未知参数

这一节做一个非常经典的 PINN 用法：不仅求解未知函数，还学习方程里的未知物理参数。

目标方程：

$$u_t = \alpha u_{xx}, \quad x \in [0,1], \quad t \in [0,1]$$

设真实扩散系数为 `alpha = 0.3`，真实解为：

$$u(x,t)=\sin(\pi x)\exp(-\alpha \pi^2 t)$$

我们只给网络少量观测点，同时要求它满足 PDE，最后看能否恢复 `alpha`。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(2)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
alpha_true = 0.3

def exact_solution(x, t, alpha=alpha_true):
    return torch.sin(math.pi * x) * torch.exp(-alpha * math.pi**2 * t)

class MLP(nn.Module):
    def __init__(self, in_dim=2, out_dim=1, hidden_dim=64, depth=4):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, xt):
        return self.net(xt)

model = MLP().to(device)
raw_alpha = nn.Parameter(torch.tensor(0.0, device=device))

def positive_alpha():
    return torch.nn.functional.softplus(raw_alpha)

## 构造数据点和物理配点

数据点来自真实解，可以理解为实验测量。物理配点只需要 `(x,t)`，不需要真实 `u`。

In [ ]:
n_data = 100
x_data = torch.rand(n_data, 1, device=device)
t_data = torch.rand(n_data, 1, device=device)
u_data = exact_solution(x_data, t_data)

# 加一点点噪声，模拟实验误差。想看理想情况可以把 0.01 改成 0。
u_data = u_data + 0.01 * torch.randn_like(u_data)

x_b = torch.cat([torch.zeros(50, 1, device=device), torch.ones(50, 1, device=device)], dim=0)
t_b = torch.rand(100, 1, device=device)
u_b = torch.zeros_like(x_b)

x_i = torch.rand(100, 1, device=device)
t_i = torch.zeros_like(x_i)
u_i = exact_solution(x_i, t_i)

In [ ]:
def heat_residual(model, x, t):
    x.requires_grad_(True)
    t.requires_grad_(True)
    xt = torch.cat([x, t], dim=1)
    u = model(xt)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t - positive_alpha() * u_xx

In [ ]:
optimizer = torch.optim.Adam(list(model.parameters()) + [raw_alpha], lr=1e-3)
history = []

for epoch in range(9000):
    x_f = torch.rand(512, 1, device=device)
    t_f = torch.rand(512, 1, device=device)
    r = heat_residual(model, x_f, t_f)
    loss_phys = torch.mean(r**2)

    loss_data = torch.mean((model(torch.cat([x_data, t_data], dim=1)) - u_data)**2)
    loss_bc = torch.mean((model(torch.cat([x_b, t_b], dim=1)) - u_b)**2)
    loss_ic = torch.mean((model(torch.cat([x_i, t_i], dim=1)) - u_i)**2)

    loss = loss_phys + 10.0 * loss_data + 5.0 * loss_bc + 5.0 * loss_ic

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    history.append([loss.item(), loss_phys.item(), loss_data.item(), positive_alpha().item()])
    if epoch % 1500 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.3e}, alpha={positive_alpha().item():.4f}")

In [ ]:
history_t = torch.tensor(history)
print(f"true alpha = {alpha_true:.4f}")
print(f"learned alpha = {positive_alpha().item():.4f}")

x_line = torch.linspace(0, 1, 300, device=device).reshape(-1, 1)
times = [0.0, 0.25, 0.5, 1.0]

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
for tv in times:
    t_line = torch.full_like(x_line, tv)
    with torch.no_grad():
        pred = model(torch.cat([x_line, t_line], dim=1))
        exact = exact_solution(x_line, t_line)
    plt.plot(x_line.cpu(), exact.cpu(), label=f"exact t={tv}")
    plt.plot(x_line.cpu(), pred.cpu(), "--")
plt.title("Solution slices")
plt.legend(fontsize=8)

plt.subplot(1, 3, 2)
plt.semilogy(history_t[:, 0], label="total")
plt.semilogy(history_t[:, 1], label="physics")
plt.semilogy(history_t[:, 2], label="data")
plt.legend()
plt.title("Loss terms")

plt.subplot(1, 3, 3)
plt.plot(history_t[:, 3])
plt.axhline(alpha_true, color="black", linestyle="--", label="true")
plt.title("Learned alpha")
plt.legend()
plt.tight_layout()
plt.show()

## 练习

1. 把数据点 `n_data` 从 `100` 改成 `20`，观察参数反演是否稳定。
2. 把噪声从 `0.01` 改成 `0.05`，观察 `alpha` 是否偏移。
3. 去掉物理损失，只用数据拟合，比较外推表现。
4. 把 `alpha_true` 改成其他正数，测试能否重新学习出来。